# LLM4OR GRPO 训练
## 运筹优化建模大语言模型强化学习训练

本 notebook 实现了基于 GRPO (Group Relative Policy Optimization) 的运筹优化建模 LLM 训练。

### 核心流程
1. **数据准备**: 下载并预处理 OptMATH/NL4OPT 数据
2. **SFT 阶段**: 指令微调，让模型学会基本格式
3. **GRPO 阶段**: 强化学习优化，提升建模能力
4. **评测**: 在基准数据集上评估模型性能

### 关键创新
- **两步训练法**: 先 SFT 打基础，再 GRPO 提升
- **多维度奖励**: 格式奖励 + 执行奖励 + 答案奖励
- **组内相对优势**: GRPO 的核心优势计算
- **修复版奖励函数**: 防止 "python" 刷分攻击

In [ ]:
# 安装依赖
!pip install -q torch transformers accelerate bitsandbytes peft datasets pyomo highspy tensorboard trl

In [ ]:
# 检查环境
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. 数据准备

In [ ]:
import json
import os
from datasets import load_dataset

def load_and_prepare_data(max_samples=5000):
    """加载并准备训练数据"""
    print("加载数据集...")
    
    # 尝试加载 OptMATH
    train_data = []
    
    try:
        ds = load_dataset('AI4Math/OptMATH', trust_remote_code=True)
        for split in ds:
            for item in ds[split]:
                problem = item.get('problem', '') or item.get('text', '')
                code = item.get('code', '') or item.get('solution', '')
                answer = item.get('answer', item.get('optimal_value', ''))
                
                if not problem or not code:
                    continue
                
                prompt = (
                    f"请为以下运筹优化问题建立数学模型并编写 Pyomo 代码求解。"
                    f"\n\n问题描述：\n{problem}"
                )
                
                train_data.append({
                    'id': f"optmath_{len(train_data)}",
                    'prompt': prompt,
                    'response': f"```python\n{code}\n```",
                    'problem_text': problem,
                    'code_solution': code,
                    'answer': str(answer) if answer else '',
                })
        print(f"OptMATH: 加载了 {len(train_data)} 条数据")
    except Exception as e:
        print(f"OptMATH 加载失败: {e}")
    
    # 采样
    if max_samples > 0 and len(train_data) > max_samples:
        import random
        random.seed(42)
        train_data = random.sample(train_data, max_samples)
        print(f"采样到 {len(train_data)} 条数据")
    
    # 保存
    os.makedirs('./data/processed', exist_ok=True)
    with open('./data/processed/train.jsonl', 'w', encoding='utf-8') as f:
        for item in train_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    
    print(f"数据已保存: {len(train_data)} 条")
    return train_data

train_data = load_and_prepare_data(max_samples=5000)

## 2. SFT 阶段训练

In [ ]:
# SFT 训练
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from torch.utils.data import Dataset as TorchDataset
import json

class ORDataset(TorchDataset):
    def __init__(self, data, tokenizer, max_length=1536):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        prompt = item.get('prompt', '')
        response = item.get('response', '')
        
        # 构建完整文本
        full_text = f"{prompt}\n\n{response}{self.tokenizer.eos_token}"
        
        encoding = self.tokenizer(
            full_text,
            max_length=self.max_length,
            truncation=True,
            padding='max_length',
            return_tensors='pt',
        )
        
        input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()
        
        # 计算 prompt 长度
        prompt_encoding = self.tokenizer(prompt, add_special_tokens=False)
        prompt_len = len(prompt_encoding['input_ids']) + 2  # +2 for \n\n
        
        # Labels: prompt 部分置 -100
        labels = input_ids.clone()
        labels[:prompt_len] = -100
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
        }

def run_sft(model_name, train_data, output_dir='./outputs/sft', num_epochs=3):
    print(f"开始 SFT 训练: {model_name}")
    
    # 加载 tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # 加载模型
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map='auto',
        trust_remote_code=True,
    )
    
    # 冻结并应用 LoRA
    for param in model.parameters():
        param.requires_grad = False
    
    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                        'gate_proj', 'up_proj', 'down_proj'],
        bias='none',
        task_type=TaskType.CAUSAL_LM,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    
    # 准备数据集
    dataset = ORDataset(train_data, tokenizer)
    
    # 训练参数
    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=num_epochs,
        learning_rate=2e-4,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        logging_steps=10,
        save_steps=200,
        save_total_limit=2,
        fp16=torch.cuda.is_available(),
        remove_unused_columns=False,
        report_to=['tensorboard'],
    )
    
    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    
    # 训练
    trainer.train()
    
    # 保存
    final_path = f"{output_dir}/final"
    trainer.save_model(final_path)
    tokenizer.save_pretrained(final_path)
    
    print(f"SFT 模型已保存: {final_path}")
    return final_path

# 运行 SFT
MODEL_NAME = "Qwen/Qwen2.5-1.5B"  # 或 "Qwen/Qwen2.5-0.5B" 用于快速测试
sft_model_path = run_sft(MODEL_NAME, train_data, num_epochs=3)

## 3. GRPO 阶段训练

In [ ]:
# 奖励函数
import re
from dataclasses import dataclass

@dataclass
class RewardResult:
    total_reward: float
    format_reward: float
    execution_reward: float
    answer_reward: float = 0.0
    is_valid: bool = False

class RewardFunction:
    """修复版奖励函数 - 防止刷分攻击"""
    
    FORMAT_ITEMS = {
        'import_pyomo': (r'import\s+pyomo|from\s+pyomo', 0.05),
        'model_def': (r'(Model|model)\s*=', 0.05),
        'variables': (r'(Var|var)\s*\(', 0.05),
        'objective': (r'(Objective|objective|expr)\s*\(', 0.05),
        'constraints': (r'(Constraint|constraint)\s*\(', 0.05),
        'solver': (r'\.solve|\.SolverFactory', 0.05),
    }
    
    def __init__(self, format_scale=1.0, exec_scale=2.0, answer_scale=3.0):
        self.format_scale = format_scale
        self.exec_scale = exec_scale
        self.answer_scale = answer_scale
    
    def extract_code(self, text):
        patterns = [
            r'```python\n(.*?)```',
            r'```pyomo\n(.*?)```',
            r'```\n(.*?)```',
        ]
        for pattern in patterns:
            match = re.search(pattern, text, re.DOTALL)
            if match:
                return match.group(1).strip()
        return None
    
    def compute_format_reward(self, text):
        code = self.extract_code(text)
        if code is None:
            return 0.0
        
        reward = 0.0
        critical_passed = 0
        
        for name, (pattern, weight) in self.FORMAT_ITEMS.items():
            if re.search(pattern, code, re.IGNORECASE):
                reward += weight
                if name in ['import_pyomo', 'model_def', 'solver']:
                    critical_passed += 1
        
        # 完整性奖励
        if critical_passed == 3:
            reward += 0.15
        
        return min(reward * self.format_scale, 1.5)
    
    def execute_code(self, code):
        """执行 Pyomo 代码"""
        import sys
        import io
        from contextlib import redirect_stdout, redirect_stderr
        
        exec_globals = {}
        try:
            from pyomo.environ import (
                ConcreteModel, Objective, Constraint, Var,
                SolverFactory, value, minimize, maximize
            )
            exec_globals.update({
                'ConcreteModel': ConcreteModel,
                'Objective': Objective,
                'Constraint': Constraint,
                'Var': Var,
                'SolverFactory': SolverFactory,
                'value': value,
                'minimize': minimize,
                'maximize': maximize,
            })
        except ImportError:
            return False, None, "Pyomo 未安装"
        
        stdout_capture = io.StringIO()
        try:
            with redirect_stdout(stdout_capture), redirect_stderr(io.StringIO()):
                exec(code, exec_globals)
            return True, None, ""
        except Exception as e:
            return False, None, str(e)
    
    def __call__(self, text, ground_truth=None, is_last=False):
        format_reward = self.compute_format_reward(text)
        
        execution_reward = 0.0
        is_valid = False
        
        # 只在格式基本正确时尝试执行
        if is_last and format_reward > 0.5:
            code = self.extract_code(text)
            if code:
                success, _, _ = self.execute_code(code)
                if success:
                    execution_reward = 1.0 * self.exec_scale
                    is_valid = True
        
        total = format_reward + execution_reward
        
        return RewardResult(
            total_reward=total,
            format_reward=format_reward,
            execution_reward=execution_reward,
            answer_reward=0.0,
            is_valid=is_valid,
        )

# 测试奖励函数
rf = RewardFunction()

# 测试 1: 刷分攻击
hack_text = "python python python python python python python"
result = rf(hack_text, is_last=True)
print(f"刷分攻击测试: reward={result.total_reward:.3f} (应为 0)")
assert result.total_reward == 0.0, "应该抵御刷分攻击！"

# 测试 2: 有效代码
valid_text = """
```python
from pyomo.environ import *
m = ConcreteModel()
m.x = Var(within=NonNegativeReals)
m.obj = Objective(expr=m.x, sense=minimize)
m.c = Constraint(expr=m.x >= 5)
SolverFactory('cbc').solve(m)
```
"""
result = rf(valid_text, is_last=True)
print(f"有效代码测试: reward={result.total_reward:.3f}")
print(f"  格式奖励: {result.format_reward:.3f}")
print(f"  执行奖励: {result.execution_reward:.3f}")
print(f"  执行状态: {'成功' if result.is_valid else '失败'}")

print("奖励函数测试通过！")

In [ ]:
# GRPO 训练
import numpy as np
from trl import GRPOTrainer, GRPOConfig
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset as HFDataset

def prepare_dataset(data, tokenizer, max_length=1536):
    """准备 HuggingFace Dataset"""
    texts = []
    answers = []
    
    for item in data:
        prompt = item.get('prompt', '')
        response = item.get('response', '')
        
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": response},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(text)
        answers.append(item.get('answer', ''))
    
    return HFDataset.from_dict({"text": texts, "answer": answers})

def run_grpo(sft_model_path, train_data, output_dir='./outputs/grpo'):
    print("开始 GRPO 训练...")
    
    # 加载 SFT 模型
    tokenizer = AutoTokenizer.from_pretrained(sft_model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        sft_model_path,
        torch_dtype=torch.float32,
        device_map='auto',
        trust_remote_code=True,
    )
    
    # 准备数据集
    dataset = prepare_dataset(train_data, tokenizer)
    
    # 奖励函数
    reward_fn = RewardFunction()
    
    def reward_function(prompts, responses, **kwargs):
        rewards = []
        for i, (prompt, response) in enumerate(zip(prompts, responses)):
            result = reward_fn(response, is_last=True)
            rewards.append(result.total_reward)
        return rewards
    
    # GRPO 配置
    grpo_config = GRPOConfig(
        output_dir=output_dir,
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=1e-5,
        lr_scheduler_type='cosine',
        warmup_ratio=0.03,
        logging_steps=10,
        save_steps=200,
        save_total_limit=2,
        max_length=1536,
        max_prompt_length=768,
        max_completion_length=768,
        num_generations=4,  # 每个样本生成 4 个
        beta=0.04,  # KL 惩罚系数
        fp16=torch.cuda.is_available(),
        report_to=['tensorboard'],
    )
    
    # 创建训练器
    trainer = GRPOTrainer(
        model=model,
        config=grpo_config,
        reward_function=reward_function,
        train_dataset=dataset,
        tokenizer=tokenizer,
    )
    
    # 训练
    trainer.train()
    
    # 保存
    final_path = f"{output_dir}/final"
    trainer.save_model(final_path)
    tokenizer.save_pretrained(final_path)
    
    print(f"GRPO 模型已保存: {final_path}")
    return final_path

# 运行 GRPO
grpo_model_path = run_grpo(sft_model_path, train_data)

## 4. 评测

In [ ]:
# 评测函数
def evaluate_model(model_path, test_data, num_samples=50):
    """评测模型"""
    from transformers import AutoTokenizer, AutoModelForCausalLM
    import torch
    
    print(f"加载模型: {model_path}")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float32,
        device_map='auto',
        trust_remote_code=True,
    )
    model.eval()
    
    reward_fn = RewardFunction()
    
    results = []
    for i, item in enumerate(test_data[:num_samples]):
        prompt = item.get('prompt', '')
        
        # 生成
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
            )
        
        response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        
        # 计算奖励
        reward_result = reward_fn(response, is_last=True)
        
        results.append({
            'id': item.get('id', f'test_{i}'),
            'reward': reward_result.total_reward,
            'format_reward': reward_result.format_reward,
            'execution_reward': reward_result.execution_reward,
            'is_valid': reward_result.is_valid,
        })
        
        print(f"[{i+1}/{num_samples}] Reward={reward_result.total_reward:.2f} "
              f"(F={reward_result.format_reward:.1f}, E={reward_result.execution_reward:.1f}) "
              f"{'✓' if reward_result.is_valid else '✗'}")
    
    # 统计
    rewards = [r['reward'] for r in results]
    valid_count = sum(1 for r in results if r['is_valid'])
    
    print(f"\n{'='*50}")
    print(f"评测结果 (n={len(results)})")
    print(f"  平均奖励: {np.mean(rewards):.3f} ± {np.std(rewards):.3f}")
    print(f"  执行率: {valid_count/len(results):.2%}")
    print(f"  最高奖励: {np.max(rewards):.3f}")
    print(f"{'='*50}")
    
    return results

# 评测 GRPO 模型
eval_results = evaluate_model(grpo_model_path, train_data, num_samples=50)

## 5. 查看训练日志

训练过程中，日志会保存到 TensorBoard。可以在 Colab 中运行以下命令查看：

```python
%load_ext tensorboard
%tensorboard --logdir=./outputs/
```